# 01 - Credit Card Fraud Dataset Loading & Exploratory Data Analysis

This notebook inspects, validates, and analyzes the public Credit Card Fraud Detection benchmark dataset.

In [1]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root in sys.path
sys.path.insert(0, os.path.abspath('..'))
from config.config import RAW_DATA_PATH

print('Loading raw dataset from:', RAW_DATA_PATH)
df = pd.read_csv(RAW_DATA_PATH)
print('Dataset shape:', df.shape)

Loading raw dataset from: /Users/yaseen/Data_Analysis/BDP Project/data/raw/creditcard.csv


Dataset shape: (284807, 31)


In [2]:
# Dataset overview and columns
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# Missing value checks
missing_counts = df.isnull().sum()
print('Total missing values in dataset:', missing_counts.sum())

Total missing values in dataset: 0


In [4]:
# Class imbalance analysis
class_counts = df['Class'].value_counts()
legit_count = class_counts[0]
fraud_count = class_counts[1]
total_count = len(df)
fraud_pct = (fraud_count / total_count) * 100

print(f'Total Transactions: {total_count:,}')
print(f'Legitimate Transactions (Class 0): {legit_count:,} ({100 - fraud_pct:.3f}%)')
print(f'Fraudulent Transactions (Class 1): {fraud_count:,} ({fraud_pct:.3f}%)')
print(f'Class Imbalance Ratio: 1 fraud per {legit_count // fraud_count:,} legitimate')

Total Transactions: 284,807
Legitimate Transactions (Class 0): 284,315 (99.827%)
Fraudulent Transactions (Class 1): 492 (0.173%)
Class Imbalance Ratio: 1 fraud per 577 legitimate


In [5]:
# Transaction Amount Distribution
print('Amount summary for Legitimate transactions:')
print(df[df['Class'] == 0]['Amount'].describe().round(2))
print('\nAmount summary for Fraud transactions:')
print(df[df['Class'] == 1]['Amount'].describe().round(2))

Amount summary for Legitimate transactions:
count    284315.00
mean         88.29
std         250.11
min           0.00
25%           5.65
50%          22.00
75%          77.05
max       25691.16
Name: Amount, dtype: float64

Amount summary for Fraud transactions:
count     492.00
mean      122.21
std       256.68
min         0.00
25%         1.00
50%         9.25
75%       105.89
max      2125.87
Name: Amount, dtype: float64


In [6]:
# Time-based sequence and temporal split demonstration
df = df.sort_values(by='Time').reset_index(drop=True)
split_idx = int(0.80 * len(df))
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]
cutoff_time = train_df['Time'].max()

print(f'Time split cutoff: {cutoff_time:.1f} seconds')
print(f'Training window:  0.0s to {train_df["Time"].max():.1f}s | Txns: {len(train_df):,} | Frauds: {(train_df["Class"] == 1).sum()}')
print(f'Testing window:   {test_df["Time"].min():.1f}s to {test_df["Time"].max():.1f}s | Txns: {len(test_df):,} | Frauds: {(test_df["Class"] == 1).sum()}')

Time split cutoff: 145247.0 seconds
Training window:  0.0s to 145247.0s | Txns: 227,845 | Frauds: 417
Testing window:   145248.0s to 172792.0s | Txns: 56,962 | Frauds: 75


## Notebook Explanation

### What was done
- Loaded the complete 284,807 transaction public credit card dataset.
- Validated schema integrity, column datatypes, and verified 0 missing values.
- Conducted comprehensive class imbalance analysis (284,315 legitimate vs 492 fraud cases, 0.173% fraud rate).
- Analyzed transaction amount statistics across legitimate vs fraudulent subsets.
- Formulated the chronological time-based split without data leakage.

### Why it was done
Exploratory data analysis is essential to understand the real-world operational challenges of credit card fraud detection: extreme class imbalance and temporal dependencies. Traditional random cross-validation introduces future leakage, making a strict chronological train/test split necessary.

### Input
- Raw transaction dataset at `data/raw/creditcard.csv`.

### Output
- Validated dataset characteristics and verified absence of missing records.
- Time split cutoff threshold established at 145,247.0 seconds (~40.3 hours).

### Connection to next stage
The raw transaction dataset is ready for distributed Spark batch processing in `02_spark_batch_processing.ipynb` to engineer distributed big-data features and store them in columnar Parquet format.

### Implementation decisions
- Chronological ordering by the `Time` feature was strictly preserved to avoid temporal leakage.
- Accuracy was explicitly disqualified as a primary metric due to the 99.827% majority class.